In [ ]:
import os
os.chdir(os.environ.get("PROJECT_ROOT", "../.."))

import pickle
from pathlib import Path
from typing import List
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

base = Path("results/expert-coactivation")
assert base.is_dir()

In [ ]:
def plot(glob: List[Path], nrows: int, ncols: int, k: int, save: Path, minimal: bool = False):
    """
    Plot coactivation heatmaps.

    Args:
        minimal: If True, remove expert ID labels and per-tile titles,
                 keep only one title per row (model), and move colorbar to right.
    """
    if minimal:
        fig, axs = plt.subplots(nrows, ncols, figsize=(24, 4 * nrows))
    else:
        fig, axs = plt.subplots(nrows, ncols, figsize=(24, 18))
    cmap = LinearSegmentedColormap.from_list("white_to_tartan", ["white", "#043673"])
    if nrows == 1 and ncols == 1:
        axs = np.array([[axs]])
    elif nrows == 1:
        axs = axs.reshape(1, -1)
    elif ncols == 1:
        axs = axs.reshape(-1, 1)

    data_records = []
    global_min, global_max = float('inf'), float('-inf')
    for file in glob:
        with file.open("rb") as f:
            record = np.array(pickle.load(f))
            expert_max = record.max(axis=1)
            top_expert_indices = np.argsort(expert_max)[-k:][::-1]
            filtered_record = record[top_expert_indices][:, top_expert_indices]
            data_records.append((file, filtered_record, top_expert_indices))
            global_min = min(global_min, filtered_record.min())
            global_max = max(global_max, filtered_record.max())

    for idx, (file, filtered_record, top_expert_indices) in enumerate(data_records):
        row, col = idx // ncols, idx % ncols
        ax = axs[row, col]

        X, Y = np.meshgrid(np.arange(k + 1), np.arange(k + 1))
        heatmap = ax.pcolormesh(X, Y, filtered_record, cmap=cmap, shading='auto', vmin=global_min, vmax=global_max)
        ax.invert_yaxis()
        ax.set_aspect("equal")

        if minimal:
            # Remove all tick labels and axis labels
            ax.set_xticks([])
            ax.set_yticks([])
            ax.set_xlabel("")
            ax.set_ylabel("")
            # Add layer label at bottom for bottom row only
            _, _, layer = file.parts[-3:]
            layer = int(layer.removesuffix(".pkl"))
            if row == nrows - 1:
                ax.set_xlabel(f"Layer {layer}", fontsize=12)
        else:
            ax.set_xticks(np.arange(k) + 0.5)
            ax.set_yticks(np.arange(k) + 0.5)
            ax.set_xticklabels(top_expert_indices, fontsize=18)
            ax.set_yticklabels(top_expert_indices, fontsize=18)
            scale, step, layer = file.parts[-3:]
            scale = scale.replace("flame-moe", "FLAME-MoE")
            scale = scale.replace("1.7b", "1.7B-10.3B")
            scale = scale.replace("721m", "721M-3.8B")
            scale = scale.replace("290m", "290M-1.3B")
            step, layer = int(step), int(layer.removesuffix(".pkl"))
            ax.set_title(f"{scale}, step-{step}, layer-{layer}", fontsize=18)

        ax.tick_params(axis='both', which='both', length=0)
        for spine in ax.spines.values(): spine.set_visible(False)

    if minimal:
        # Add model labels on the left using fig.text (outside subplots)
        model_labels = ["Baseline", "DRMoET"]
        for row in range(nrows):
            # Calculate y position for each row center
            y_pos = 1 - (row + 0.5) / nrows
            fig.text(0.02, y_pos, model_labels[row], fontsize=14, fontweight='bold', 
                    va='center', ha='left', rotation=90)
        plt.subplots_adjust(left=0.06, right=0.92, wspace=0.05, hspace=0.15)
        # Colorbar on the right side (shorter)
        cbar = fig.colorbar(heatmap, ax=axs.ravel().tolist(), location='right', 
                           pad=0.02, fraction=0.015, aspect=20, shrink=0.7)
    else:
        plt.tight_layout()
        cbar = fig.colorbar(heatmap, ax=axs.ravel().tolist(), pad=0.02, fraction=0.01)
    cbar.ax.tick_params(labelsize=12)

    save.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(save, bbox_inches='tight', pad_inches=0.1)
    plt.close()

In [ ]:
print("across scales, last checkpoint, at the first layer")
glob = [*base.glob("*290m/5473/2.pkl"), *base.glob("*721m/8815/2.pkl"), *base.glob("*1.7b/11029/2.pkl")]
plot(glob, 1, 3, 16, Path("figures/cross-scale-expert-coactivation/last-checkpoint-first-layer.pdf"))

print("across scales, last checkpoint, at the final layer")
glob = [*base.glob("*290m/5473/9.pkl"), *base.glob("*721m/8815/12.pkl"), *base.glob("*1.7b/11029/18.pkl")]
plot(glob, 1, 3, 16, Path("figures/cross-scale-expert-coactivation/last-checkpoint-final-layer.pdf"))


In [ ]:
# Configuration for 32-expert models comparison
models = {
    "290m-seed3407-32e": "Baseline (seed3407)",
    "290m-dro-activation-beta0.999-eta0.001-alpha1.0-norml2_fine_grained_32e": "DRO Activation"
}
checkpoint = 16000
layers = [2, 3, 4, 5, 6, 7, 8, 9]

# Create output directory
output_dir = Path("figures/expert-coactivation-32e")
output_dir.mkdir(parents=True, exist_ok=True)

# Plot 1: Each model separately - all layers at checkpoint 16000
print("=== Generating per-model coactivation heatmaps ===")
for model_name, label in models.items():
    glob = []
    for layer in layers:
        pkl_path = base / model_name / str(checkpoint) / f"{layer}.pkl"
        if pkl_path.exists():
            glob.append(pkl_path)
        else:
            print(f"Missing: {pkl_path}")
    
    if len(glob) == 8:
        print(f"Plotting {label}: checkpoint {checkpoint}, {len(glob)} layers...")
        safe_name = model_name.replace("/", "_")
        plot(glob, 1, 8, 16, output_dir / f"{safe_name}-checkpoint-{checkpoint}.pdf")
    else:
        print(f"Found {len(glob)}/8 files for {label}")

# Plot 2: Compare both models side by side for each layer
print("\n=== Generating model comparison heatmaps ===")
for layer in layers:
    glob = []
    for model_name in models.keys():
        pkl_path = base / model_name / str(checkpoint) / f"{layer}.pkl"
        if pkl_path.exists():
            glob.append(pkl_path)
    
    if len(glob) == 2:
        print(f"Comparing layer {layer}...")
        plot(glob, 1, 2, 16, output_dir / f"comparison-layer-{layer}-checkpoint-{checkpoint}.pdf")

# Plot 3: All layers, both models - 2 rows (models) x 8 cols (layers)
# Same layer in same column for easy comparison
# Uses minimal=True for cleaner output (no expert IDs, one title per model)
print("\n=== Generating combined comparison (2 models x 8 layers) ===")
glob = []
model_names = list(models.keys())
for model_name in model_names:  # Row by row (model by model)
    for layer in layers:  # Column by column (layer by layer)
        pkl_path = base / model_name / str(checkpoint) / f"{layer}.pkl"
        if pkl_path.exists():
            glob.append(pkl_path)
        else:
            print(f"Missing: {pkl_path}")

if len(glob) == 16:  # 2 models x 8 layers
    print(f"Plotting combined comparison: 2 models x 8 layers...")
    plot(glob, 2, 8, 16, output_dir / f"combined-comparison-checkpoint-{checkpoint}.pdf", minimal=True)
else:
    print(f"Found {len(glob)}/16 files")

print(f"\n=== Done! Output saved to: {output_dir} ===")